<a href="https://colab.research.google.com/github/HLZHarry/Alpha-Lens-TPM/blob/main/02_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Feature Engineering


1.   Log Returns: The standard way to measure price changes in quant finance.
2.   Momentum: Is the stock trending up or down?
3.  Volatility: How "risky" has the stock been lately?
4. Technical Strength (RSI): relateive strangth index = average Gain / average Loss. Measures the speed and chagne of price movements on a scale of 0 to 100.
5. Trend Relative Value (Price vs. MA200): 200 Day moving average.


In [2]:
import pandas as pd
import numpy as np
from google.colab import drive

In [3]:
#1. Load the data from Phase 1
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Alpha-Lens-Project"
df = pd.read_csv(f"{path}/raw_market_data.csv")
df['Date'] = pd.to_datetime(df['Date'])

Mounted at /content/drive


In [4]:
# Feature Engineering
# We need to calculate features PER TICKER.
# We use groupby('Ticker') to ensure AAPL's data doesn't mix with MSFT's.

# 1. Log Returns (Better for statistical modeling)
df['Log_Return'] = df.groupby('Ticker')['Price'].transform(lambda x: np.log(x / x.shift(1)))

# 2. Momentum (Return over the last 21 trading days ~ 1 month)
df['Momentum_1M'] = df.groupby('Ticker')['Log_Return'].transform(lambda x: x.rolling(window=21).sum())

# 3. Volatility (Standard Deviation of returns over 21 ydays)
df['Vol_1M'] = df.groupby('Ticker')['Log_Return'].transform(lambda x: x.rolling(window=21).std())

#4. Target Variable: Next Day's Return (What the model will try to predict)
df['Target_Next_Return'] = df.groupby('Ticker')['Log_Return'].shift(-1)

# Drop rows with NaN values created by rolling windows/shifts
# df_features = df.dropna()

In [6]:
# Enhanced Feature Engineering
# 1. RSI (Relative Strength Index) - Identifies 'Overbought' > 70 or Oversold <30 conditions
def calculate_rsi (series, period = 14):
  delta = series.diff()
  gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
  loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
  rs = gain / loss
  return 100 - (100/(1 + rs))

df['RSI'] = df.groupby('Ticker')['Price'].transform(lambda x: calculate_rsi(x))

In [9]:
# 2. Price vs. 200-Day Moving Average
# A common 'Institutional' signal for long-term trend health
df['MA200'] = df.groupby('Ticker')['Price'].transform(lambda x: x.rolling(window=200).mean())
df['Price_to_MA200'] = df['Price'] / df['MA200']

In [10]:
# 3. Daily Range (Intraday Volatility Proxy)
# In out yfinance data, we can also use High/Low
# we use the 5-day variance of returns
df['Intraday_Var'] = df.groupby('Ticker')['Log_Return'].transform(lambda x: x.rolling(window=5).var())

In [12]:
# Cleanup and Save
df_final = df.dropna()
df_final.to_csv(f"{path}/engineered_features_v2.csv", index=False)
print(f"Phase 2 COmplete. New features count: {len(df_final.columns) - 3}")
df_final.head()

Phase 2 COmplete. New features count: 8


,Date,Ticker,Price,Log_Return,Momentum_1M,Vol_1M,Target_Next_Return,RSI,MA200,Price_to_MA200,Intraday_Var
2781,2021-10-18,AAPL,143.195755,0.011737,0.003350,0.012911,0.014967,62.159408,132.270017,1.082602,0.000141
2784,2021-10-18,CVX,91.270325,-0.000456,0.124238,0.013860,0.019344,72.431116,82.952576,1.100271,0.000019
2785,2021-10-18,GOOGL,141.707291,0.009924,0.013950,0.015363,0.003210,64.660737,117.850202,1.202436,0.000249
2786,2021-10-18,GS,369.896729,0.018591,0.055234,0.019071,-0.003705,66.254435,316.413994,1.169028,0.000223
2787,2021-10-18,JNJ,140.744308,-0.007342,-0.028506,0.007505,0.023150,41.882831,144.384060,0.974791,0.000123


1. Why Log Returns?
In institutional research, log returns are the standard for three reasons:

Time Additivity: Multi-day returns are calculated by summing daily log returns, whereas simple returns require multiplication.

Numerical Stability: They are symmetric (a gain and loss of equal magnitude return to zero), which helps ML models converge faster.

Stationarity: Log returns normalize price data, preventing "mathematical floors" that can confuse models.

2. Why shift(-1) for the Target?
This is the "Labeling" step for supervised learning:

The Goal: Predict the return for Tomorrow (t+1) using information available Today (t).

The Method: We "pull" tomorrow's actual return into today's row so the model can associate current features with the future outcome during the training phase.

3. Why groupby('Ticker')?
Since our data is in "Long Format" (multiple stocks in one table), we use groupby as a Data Shield:

Asset Isolation: It ensures calculations for Apple don't accidentally use data from Microsoft.

Bias Prevention: It stops "Cross-Ticker Leakage," ensuring the timeline for each asset remains independent.

4. Why transform()?
transform() is the engine of our feature engineering:

Shape Preservation: Unlike a standard average, transform() ensures the output is the same length as our input.

Mapping: It allows us to calculate a "group-level" metric (like Volatility) and map it back to every specific row for that stock.